# **Tugas 2: Progress Final Project SC Data Science Basic**

Market Prioritization untuk UMKM: Menentukan Segmen dan Wilayah Target Sourcing Leads B2B SaaS

**Kelompok 2**

Anggota Kelompok:
1. Aliifah Sava Fikrana - [tugas]
2. Muhammad Farras Ashshiddiqi [tugas]
3. Rama Rifqi Saputra [tugas]

## Ringkasan Sumber Data

| Sumber | File | Alasan Pemilihan | Level Data | Kolom Kunci | Keterbatasan |
|---|---|---|---|---|---|
| Open Data Jabar – Dinas Koperasi dan Usaha Kecil | 01_umkm_proyeksi_2017-2023 | Jumlah UMKM per kab/kota untuk indikator Ukuran Pasar | Kab/kota (27) | kode_kabupaten_kota, kategori_usaha, proyeksi_jumlah_umkm, tahun | Angka berupa proyeksi, bukan sensus |
| Open Data Jabar – Dinas Koperasi dan Usaha Kecil | 02_umkm_binaan_2019-2023 | Rincian jenis usaha per wilayah untuk Kepadatan Sektor Target | Kab/kota (27) | kode_kabupaten_kota, jenis_usaha, jumlah_umkm, tahun | Hanya UMKM binaan; F&B tercatat 0 di 2023 (anomali) |
| BPS | 03_internet_bps_2023 | Persentase penduduk pengguna internet untuk Kesiapan Digital | Kab/kota (27) | kode_kabupaten_kota, pct_penduduk_internet_5plus_2023 | Berbasis survei sampel (Susenas) |
| Open Data Jabar – Bapenda | 04_esamsat_2014-2025 | Nominal pembayaran digital untuk proksi Adopsi Pembayaran Digital | Kab/kota (cabang pelayanan × bank) | kode_kabupaten_kota, jumlah_pembayaran, tahun | Proksi, bukan data QRIS UMKM; kode wilayah tidak konsisten |
| BPS | 05_pdrb_perkapita_2017-2025 | PDRB per kapita untuk Aktivitas Ekonomi | Kab/kota (27) | kode_kabupaten_kota, pdrb_adhb_per_kapita, tahun | Angka tahun terbaru masih sementara |

## Bagian 1 — Data Collection & Data Understanding

### Poin 1: Pengumpulan Data Publik

Digunakan 5 dataset dari 4 sumber publik resmi: Open Data Jabar – Dinas Koperasi dan Usaha Kecil, Open Data Jabar – Bapenda, Open Data Jabar – dataset BPS (PDRB per kapita), dan publikasi BPS “Statistik Kesejahteraan Rakyat Provinsi Jawa Barat 2023” — sesuai batas 2–4 sumber. Seluruh data berada di level kabupaten/kota dan mencakup **27 wilayah Jawa Barat**, sehingga memenuhi syarat minimal 20 unit wilayah. Rincian sumber ada pada tabel ringkasan di bawah.

Keterbatasan utama: tahun data antar sumber tidak seragam (UMKM 2023, internet 2023, e-Samsat 2024, PDRB 2024), dan indikator internet hanya tersedia dari publikasi PDF BPS karena tidak ada dataset API-nya di portal data.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, pandas as pd

BASE = "/content/drive/MyDrive/Colab Notebooks/Tugas2_DS_Kelompok02"
D = f"{BASE}/data"

umkm     = pd.read_csv(f"{D}/01_umkm_proyeksi_2017-2023_raw.csv")
binaan   = pd.read_csv(f"{D}/02_umkm_binaan_2019-2023_raw.csv")
internet = pd.read_csv(f"{D}/03_internet_bps_2023_raw.csv")
esamsat  = pd.read_csv(f"{D}/04_esamsat_2014-2025_raw.csv")
pdrb     = pd.read_csv(f"{D}/05_pdrb_perkapita_2017-2025_raw.csv")

for d, n in [(umkm,"umkm"), (binaan,"binaan"), (internet,"internet"), (esamsat,"esamsat"), (pdrb,"pdrb")]:
    print(n, d.shape)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
umkm (1890, 9)
binaan (2025, 9)
internet (27, 3)
esamsat (2005, 10)
pdrb (243, 8)


## Bagian 2 — Data Preparation & Kriteria Penggugur

### Poin 1: Pembersihan Data

1. **Missing values:** terdapat 2 nilai kosong di kolom `kode_provinsi` (1 di dataset
   UMKM proyeksi, 1 di dataset e-Samsat). Karena seluruh data berasal dari Jawa Barat,
   nilai kosong diisi `32` — tidak dihapus — agar tidak ada wilayah yang hilang dari
   analisis (baris yang kosong merupakan data UMKM Kota Banjar, 4.457 unit).
2. **Duplikat:** tidak ditemukan baris duplikat di kelima dataset (dicek dengan
   `duplicated().sum()` per dataset, hasil 0).
3. **Inkonsistensi format penamaan wilayah:** nama kab/kota diseragamkan ke huruf
   kapital tanpa spasi berlebih. Data e-Samsat memuat 29 kode wilayah untuk 27 wilayah
   karena kode bergeser pada sebagian tahun (kode ekstra: [ISI — output sel cek kode],
   muncul hanya pada tahun [ISI]); kode kemudian diseragamkan menggunakan kode BPS dari
   data UMKM berdasarkan nama wilayah. Hasil: seluruh baris e-Samsat berhasil dipetakan
   ke 27 kode BPS, 0 baris tanpa kode.
4. **Data anomali:** jumlah UMKM binaan sektor F&B tercatat 0 pada 2023, padahal pada
   2019–2022 berkisar 1.600–3.600 unit. Ini dicatat sebagai anomali pencatatan dan
   keterbatasan data. Konsekuensinya, kepadatan sektor target 2023 dihitung tanpa
   kontribusi F&B, sehingga nilainya cenderung *underestimate* untuk wilayah dengan
   basis kuliner kuat.

In [ ]:
# PEMBERSIHAN DATA

data = [(umkm, "umkm"), (binaan, "binaan"), (internet, "internet"), (esamsat, "esamsat"), (pdrb, "pdrb")]

# Missing values
print("Jumlah nilai kosong (sebelum cleaning):", {n: int(d.isna().sum().sum()) for d, n in data})
for d in (umkm, binaan, esamsat, pdrb):
    d["kode_provinsi"] = d["kode_provinsi"].fillna(32)
print("Jumlah nilai kosong (setelah cleaning):", {n: int(d.isna().sum().sum()) for d, n in data})

# Duplikat
print("\nJumlah baris duplikat:", {n: int(d.duplicated().sum()) for d, n in data})
for d, n in data:
    assert d.isna().sum().sum() == 0, f"Ada nilai kosong di {n}"
    assert d.duplicated().sum() == 0, f"Ada baris yang identik di {n}"

# Inkonsistensi penamaan wilayah
for d in (umkm, binaan, internet, esamsat, pdrb):
    d["nama_kabupaten_kota"] = d["nama_kabupaten_kota"].str.strip().str.upper()

print("\nJumlah kode wilayah e-Samsat (sebelum perbaikan):", esamsat["kode_kabupaten_kota"].nunique())
ref = umkm[["kode_kabupaten_kota", "nama_kabupaten_kota"]].drop_duplicates()
esamsat = esamsat.drop(columns="kode_kabupaten_kota").merge(ref, on="nama_kabupaten_kota", how="left")
print("Jumlah kode wilayah e-Samsat (setelah perbaikan):", esamsat["kode_kabupaten_kota"].nunique())

# Data anomali
fnb_per_tahun = (binaan[binaan["jenis_usaha"].isin(["KULINER", "MAKANAN", "MINUMAN"])]
                 .groupby("tahun")["jumlah_umkm"].sum())
print("\nJumlah UMKM binaan F&B per tahun:")
print(fnb_per_tahun.to_string())

print("\nJumlah baris & kolom setelah cleaning:", umkm.shape, binaan.shape, internet.shape, esamsat.shape, pdrb.shape)
print("\nPembersihan data selesai: Tidak ada nilai kosong dan tidak ada data yang duplikat.")

Jumlah nilai kosong (sebelum cleaning): {'umkm': 1, 'binaan': 0, 'internet': 0, 'esamsat': 1, 'pdrb': 0}
Jumlah nilai kosong (setelah cleaning): {'umkm': 0, 'binaan': 0, 'internet': 0, 'esamsat': 0, 'pdrb': 0}

Jumlah baris duplikat: {'umkm': 0, 'binaan': 0, 'internet': 0, 'esamsat': 0, 'pdrb': 0}

Jumlah kode wilayah e-Samsat (sebelum perbaikan): 29
Jumlah kode wilayah e-Samsat (setelah perbaikan): 27

Jumlah UMKM binaan F&B per tahun:
tahun
2019    1619
2020    3632
2021    2616
2022    2739
2023       0

Jumlah baris & kolom setelah cleaning: (1890, 9) (2025, 9) (27, 3) (2005, 10) (243, 8)

Pembersihan data selesai: Tidak ada nilai kosong dan tidak ada data yang duplikat.


### Bagian 2 — Poin 2a: Persiapan Indikator (Ukuran Pasar & Kepadatan Sektor Target)

- **Ukuran Pasar:** total UMKM 2023 dijumlahkan per kab/kota. Pengecekan outlier dengan metode IQR tidak menemukan outlier.
- **Pemetaan sektor ke kelompok ICP Klay:** Ritel = FASHION, KONVEKSI; F&B = KULINER, MAKANAN, MINUMAN; Jasa = JASA. Jenis usaha lain (mebel, batik, agribisnis, dll.) dianggap di luar ICP karena bukan usaha dengan transaksi harian rutin.
- **Kepadatan Sektor Target:** proporsi UMKM binaan sektor ICP terhadap total UMKM binaan per wilayah (rata-rata ±84%, rentang 75%–94%).

In [ ]:
umkm_2023 = umkm[umkm["tahun"] == 2023].groupby("nama_kabupaten_kota")["proyeksi_jumlah_umkm"].sum().reset_index(name="total_umkm_2023")

q1, q3 = umkm_2023["total_umkm_2023"].quantile([0.25, 0.75])
outliers = umkm_2023[umkm_2023.total_umkm_2023 > q3 + 1.5 * (q3 - q1)]["nama_kabupaten_kota"].tolist()
print("Outlier IQR total UMKM 2023:", outliers or "tidak ada")

RITEL, FNB, JASA = ["FASHION", "KONVEKSI"], ["KULINER", "MAKANAN", "MINUMAN"], ["JASA"]

share = (
    binaan.assign(icp=binaan["jenis_usaha"].isin(RITEL + FNB + JASA))
    .groupby(["kode_kabupaten_kota", "nama_kabupaten_kota"])
    .apply(lambda g: g.loc[g["icp"], "jumlah_umkm"].sum() / g["jumlah_umkm"].sum(), include_groups=False)
    .reset_index(name="kepadatan_icp")
)

assert share["kepadatan_icp"].between(0, 1).all()
print(share["kepadatan_icp"].describe().to_string())

Outlier IQR total UMKM 2023: tidak ada
count    27.000000
mean      0.837645
std       0.044196
min       0.748865
25%       0.817096
50%       0.834731
75%       0.858312
max       0.938326


### Bagian 2 — Poin 2b: Penggabungan Data (Merging)

Kelima indikator digabung per wilayah menggunakan kunci `kode_kabupaten_kota`; e-Samsat diagregasi dulu (jumlah per kab/kota) dan dibuat variabel turunan `esamsat_per_umkm` karena nominal mentah mencerminkan ukuran wilayah, bukan perilaku digital.

In [ ]:
# PENGGABUNGAN DATA (MERGING)

print("Jumlah baris data mentah:",
      {"umkm": len(umkm), "binaan": len(binaan), "internet": len(internet),
       "esamsat": len(esamsat), "pdrb": len(pdrb)})

# Penyamaan level data: semua dataset sudah level kab/kota (27 wilayah),
# sehingga cukup difilter ke 1 tahun dan diringkas menjadi 1 baris per wilayah
pasar = umkm_2023.merge(ref, on="nama_kabupaten_kota", how="left")   # Ukuran Pasar (2023)

internet_2023 = internet[["kode_kabupaten_kota", "pct_penduduk_internet_5plus_2023"]] \
    .rename(columns={"pct_penduduk_internet_5plus_2023": "internet_pct_2023"})   # Kesiapan Digital (2023)

esamsat_2024 = (esamsat[esamsat["tahun"] == 2024]
                .groupby("kode_kabupaten_kota", as_index=False)["jumlah_pembayaran"].sum()
                .rename(columns={"jumlah_pembayaran": "esamsat_rp_2024"}))   # Adopsi Pembayaran Digital (2024)

pdrb_2024 = pdrb[pdrb["tahun"] == 2024][["kode_kabupaten_kota", "pdrb_adhb_per_kapita"]] \
    .rename(columns={"pdrb_adhb_per_kapita": "pdrb_kapita_2024"})   # Aktivitas Ekonomi (2024)

print("Jumlah baris per dataset (sebelum merging):",
      {"umkm": len(pasar), "kepadatan": len(share), "internet": len(internet_2023),
       "esamsat": len(esamsat_2024), "pdrb": len(pdrb_2024)})

# Merging: left join dengan kunci kode wilayah BPS
df = (pasar
      .merge(share.drop(columns="nama_kabupaten_kota"), on="kode_kabupaten_kota", how="left")
      .merge(internet_2023, on="kode_kabupaten_kota", how="left")
      .merge(esamsat_2024,  on="kode_kabupaten_kota", how="left")
      .merge(pdrb_2024,     on="kode_kabupaten_kota", how="left"))

# Adopsi pembayaran digital per UMKM (agar tidak bias ukuran wilayah)
df["esamsat_per_umkm"] = df["esamsat_rp_2024"] / df["total_umkm_2023"]

df = df[["kode_kabupaten_kota", "nama_kabupaten_kota", "total_umkm_2023", "kepadatan_icp",
         "internet_pct_2023", "esamsat_rp_2024", "esamsat_per_umkm", "pdrb_kapita_2024"]]
df = df.sort_values("kode_kabupaten_kota").reset_index(drop=True)

# Validasi hasil merging
assert len(df) == 27, "Jumlah wilayah tidak 27"
assert df.isna().sum().sum() == 0, "Ada nilai kosong setelah merging"
print("Ukuran tabel gabungan (setelah merging):", df.shape)
print("Jumlah nilai kosong (setelah merging):", df.isna().sum().sum())
df.head()

Jumlah baris data mentah: {'umkm': 1890, 'binaan': 2025, 'internet': 27, 'esamsat': 2005, 'pdrb': 243}
Jumlah baris per dataset (sebelum merging): {'umkm': 27, 'kepadatan': 27, 'internet': 27, 'esamsat': 27, 'pdrb': 27}
Ukuran tabel gabungan (setelah merging): (27, 8)
Jumlah nilai kosong (setelah merging): 0


,kode_kabupaten_kota,nama_kabupaten_kota,total_umkm_2023,kepadatan_icp,internet_pct_2023,esamsat_rp_2024,esamsat_per_umkm,pdrb_kapita_2024
0,3201,KABUPATEN BOGOR,570944,0.766164,75.11,95722879200,167657.211916,54859
1,3202,KABUPATEN SUKABUMI,409507,0.782051,63.69,12700871000,31015.027826,31167
2,3203,KABUPATEN CIANJUR,381810,0.820408,65.02,13978727900,36611.738561,24914
3,3204,KABUPATEN BANDUNG,537801,0.884043,74.97,52164661800,96996.215701,44126
4,3205,KABUPATEN GARUT,394494,0.834731,61.55,11912313200,30196.436955,29012


### Bagian 2 — Poin 3: Penyaringan Kriteria Penggugur

Kriteria mengacu pada Brief ICP Klay Bagian 4: (1) wilayah dengan UMKM sangat kecil — di bawah persentil ke-10; (2) data kunci kosong; (3) sektor tanpa transaksi harian rutin — sudah dikeluarkan sejak pemetaan sektor.

In [ ]:
# PENYARINGAN KRITERIA PENGGUGUR

# Kriteria 1: jumlah UMKM sangat kecil (di bawah persentil ke-10)
batas_umkm = df["total_umkm_2023"].quantile(0.10)
print(f"Batas minimum jumlah UMKM (persentil ke-10): {batas_umkm:,.0f} unit")

# Kriteria 2: data kunci kosong
kosong = df.isna().any(axis=1)
print("Jumlah wilayah dengan data kunci kosong:", int(kosong.sum()))

# Daftar wilayah tereliminasi beserta alasannya
eliminasi = df[(df["total_umkm_2023"] < batas_umkm) | kosong].copy()
eliminasi["alasan"] = eliminasi.apply(
    lambda r: "Data kunci kosong" if r.isna().any()
    else f"Jumlah UMKM sangat kecil ({r['total_umkm_2023']:,.0f} < batas {batas_umkm:,.0f})", axis=1)

# Wilayah yang lolos
df_final = df.drop(eliminasi.index).reset_index(drop=True)

assert len(df_final) >= 20, "Wilayah tersisa kurang dari 20"
print(f"\nWilayah awal: {len(df)} | Tereliminasi: {len(eliminasi)} | Lolos: {len(df_final)}")
print("\nDaftar wilayah tereliminasi:")
eliminasi[["kode_kabupaten_kota", "nama_kabupaten_kota", "total_umkm_2023", "alasan"]]

Batas minimum jumlah UMKM (persentil ke-10): 76,475 unit
Jumlah wilayah dengan data kunci kosong: 0

Wilayah awal: 27 | Tereliminasi: 3 | Lolos: 24

Daftar wilayah tereliminasi:


,kode_kabupaten_kota,nama_kabupaten_kota,total_umkm_2023,alasan
19,3272,KOTA SUKABUMI,60866,"Jumlah UMKM sangat kecil (60,866 < batas 76,475)"
21,3274,KOTA CIREBON,61234,"Jumlah UMKM sangat kecil (61,234 < batas 76,475)"
26,3279,KOTA BANJAR,39422,"Jumlah UMKM sangat kecil (39,422 < batas 76,475)"


### Bagian 2 — Poin 4: Penyamaan Skala (Normalisasi)

Min-Max Scaling ke rentang 0–1 karena satuan antar indikator berbeda; seluruh indikator berarah positif sesuai ICP sehingga tidak ada inversi.

In [23]:
# POIN 4: NORMALISASI MIN-MAX (0-1)
from sklearn.preprocessing import MinMaxScaler

INDIKATOR = ["total_umkm_2023", "kepadatan_icp", "internet_pct_2023",
             "esamsat_per_umkm", "pdrb_kapita_2024"]

assert df_final[INDIKATOR].isna().sum().sum() == 0, "ada missing - perbaiki cleaning dulu"

scaled = pd.DataFrame(MinMaxScaler().fit_transform(df_final[INDIKATOR]),
                      columns=[c + "_scaled" for c in INDIKATOR])

df_scaled = pd.concat([df_final[["kode_kabupaten_kota", "nama_kabupaten_kota"]]
                       .reset_index(drop=True), scaled], axis=1)

assert df_scaled.isna().sum().sum() == 0
assert len(df_scaled) == len(df_final) == df_final["kode_kabupaten_kota"].nunique()

os.makedirs(f"{BASE}/output", exist_ok=True)
df_scaled.to_csv(f"{BASE}/output/data_scaled.csv", index=False)
print(f"data_scaled.csv tersimpan | {len(df_scaled)} wilayah x {len(INDIKATOR)} indikator")
df_scaled.head()

data_scaled.csv tersimpan | 24 wilayah x 5 indikator


,kode_kabupaten_kota,nama_kabupaten_kota,total_umkm_2023_scaled,kepadatan_icp_scaled,internet_pct_2023_scaled,esamsat_per_umkm_scaled,pdrb_kapita_2024_scaled
0,3201,KABUPATEN BOGOR,1.000000,0.091304,0.533715,0.327346,0.245115
1,3202,KABUPATEN SUKABUMI,0.666665,0.175160,0.148685,0.022824,0.051184
2,3203,KABUPATEN CIANJUR,0.609476,0.377613,0.193527,0.035297,0.000000
3,3204,KABUPATEN BANDUNG,0.931566,0.713484,0.528995,0.169871,0.157260
4,3205,KABUPATEN GARUT,0.635666,0.453209,0.076534,0.021000,0.033544


## Bagian 3 — EDA & Visualisasi

Lima visualisasi (univariat, bivariat, multivariat) atas 24 wilayah yang lolos kriteria penggugur.

### Interpretasi Bisnis

**G1 — Distribusi ukuran pasar (univariat).** Sebaran total UMKM antar 24 wilayah condong ke kanan (skewness 0,45): mayoritas wilayah berada di kisaran 200–400 ribu unit, sementara tiga wilayah terbesar — Kabupaten Bogor (571 ribu), Kabupaten Bandung (538 ribu), dan Kota Bandung (524 ribu) — menyendiri di atas 500 ribu. Artinya, volume calon leads terkonsentrasi pada segelintir wilayah; SDR yang menginginkan dampak cepat pada pipeline akan mendapat hasil paling besar dari beberapa wilayah ini. Tidak ditemukan outlier IQR, menunjukkan konsentrasi ini karakteristik pasar, bukan kesalahan data.

**G2 — Kepadatan sektor target (univariat).** Seluruh wilayah memiliki share sektor ICP di atas 74% (terendah Kabupaten Subang 74,9%, tertinggi Kabupaten Kuningan 93,8%). Usaha bertipe transaksi harian rutin (ritel, F&B, jasa) dominan di semua wilayah, sehingga keberadaan segmen sasaran bukan pembeda antar wilayah — yang membedakan hanya proporsinya. Keterbatasan: nilai 2023 dihitung tanpa kontribusi F&B karena anomali pencatatan (F&B tercatat 0 pada 2023), sehingga kepadatan cenderung underestimate untuk wilayah dengan basis kuliner kuat.

**G3 — Kesiapan digital vs ukuran pasar (bivariat).** Korelasi hampir nol (r = 0,02): wilayah dengan penetrasi internet tinggi tidak otomatis memiliki banyak UMKM. Kabupaten dengan pasar sangat besar justru punya penetrasi internet menengah, sementara kota-kota kecil unggul digital tetapi kecil pasarnya. Implikasi: kesiapan digital dan ukuran pasar mengukur hal yang berbeda, sehingga keduanya memang layak menjadi komponen terpisah dalam skor prioritas — wilayah ideal bagi Klay adalah yang menggabungkan keduanya.

**G4 — Aktivitas ekonomi vs adopsi pembayaran digital (bivariat).** Korelasi moderat positif (r = 0,41): masyarakat berdaya beli tinggi cenderung lebih banyak bertransaksi non-tunai melalui e-Samsat, dengan Kota Depok dan Kota Bekasi sebagai wilayah adopsi tertinggi per UMKM. Namun sebarannya cukup luas sehingga daya beli bukan penentu tunggal perilaku digital — konsisten dengan posisinya sebagai komponen pendukung (bobot terkecil) dalam ICP.

**G5 — Matriks korelasi antar indikator (multivariat).** Dua temuan: (1) korelasi internet–e-Samsat per UMKM sangat tinggi (r = 0,93) — proksi e-Samsat sebagian besar menangkap kesiapan digital wilayah, bukan murni perilaku UMKM; bobot kedua komponen ini (20% + 15%) sebagian menghitung hal serupa, dan hal ini dicatat sebagai keterbatasan serta bahan uji sensitivitas bobot pada Final Project; (2) tidak ada korelasi lain di atas 0,45, dan kepadatan sektor justru sedikit berbanding terbalik dengan ukuran pasar (r = −0,29) — pasar besar cenderung sedikit lebih rendah porsi sektor ICP-nya, sehingga prioritas tidak dapat ditentukan hanya oleh ukuran wilayah.

In [ ]:
# BAGIAN 3: EDA (5 VISUALISASI)
import matplotlib.pyplot as plt
import numpy as np

os.makedirs(f"{BASE}/assets", exist_ok=True)

# G1 univariat: distribusi jumlah UMKM
fig, a = plt.subplots(1, 2, figsize=(11, 4))
a[0].hist(df_final["total_umkm_2023"] / 1000, bins=10, color="teal", edgecolor="white")
a[0].set_xlabel("Total UMKM 2023 (ribu)"); a[0].set_title("Distribusi ukuran pasar")
a[1].boxplot(df_final["total_umkm_2023"] / 1000, vert=False)
a[1].set_title("Boxplot ukuran pasar")
plt.tight_layout(); plt.savefig(f"{BASE}/assets/g1_distribusi_umkm.png", dpi=150); plt.show()
print("skewness:", round(df_final["total_umkm_2023"].skew(), 2))
print(df_final.nlargest(3, "total_umkm_2023")[["nama_kabupaten_kota", "total_umkm_2023"]].to_string(index=False))

# G2 univariat: kepadatan sektor target per wilayah
s = df_final.sort_values("kepadatan_icp").set_index("nama_kabupaten_kota")["kepadatan_icp"]
fig, a = plt.subplots(figsize=(7, 6))
a.barh(s.index, s * 100, color="seagreen")
a.set_xlabel("Share sektor ICP (%)"); a.set_title("Kepadatan sektor target ICP per wilayah")
plt.tight_layout(); plt.savefig(f"{BASE}/assets/g2_kepadatan_icp.png", dpi=150); plt.show()
print("min:", s.index[0], round(s.iloc[0]*100, 1), "% | max:", s.index[-1], round(s.iloc[-1]*100, 1), "%")

# G3 bivariat: kesiapan digital vs ukuran pasar
fig, a = plt.subplots(figsize=(6, 4.5))
a.scatter(df_final["internet_pct_2023"], df_final["total_umkm_2023"] / 1000, s=60, alpha=0.8)
z = np.polyfit(df_final["internet_pct_2023"], df_final["total_umkm_2023"], 1)
xs = np.linspace(df_final["internet_pct_2023"].min(), df_final["internet_pct_2023"].max(), 50)
a.plot(xs, np.polyval(z, xs), "r--")
a.set_xlabel("% pengguna internet 2023"); a.set_ylabel("Total UMKM (ribu)")
a.set_title("Kesiapan digital vs ukuran pasar")
plt.tight_layout(); plt.savefig(f"{BASE}/assets/g3_internet_vs_pasar.png", dpi=150); plt.show()
print("korelasi internet-UMKM:", round(df_final["internet_pct_2023"].corr(df_final["total_umkm_2023"]), 2))

# G4 bivariat: aktivitas ekonomi vs adopsi pembayaran digital
fig, a = plt.subplots(figsize=(6, 4.5))
a.scatter(df_final["pdrb_kapita_2024"], df_final["esamsat_per_umkm"] / 1e6, s=60, alpha=0.8, color="darkorange")
z = np.polyfit(df_final["pdrb_kapita_2024"], df_final["esamsat_per_umkm"], 1)
xs = np.linspace(df_final["pdrb_kapita_2024"].min(), df_final["pdrb_kapita_2024"].max(), 50)
a.plot(xs, np.polyval(z, xs), "r--")
a.set_xlabel("PDRB/kapita 2024 (ribu Rp)"); a.set_ylabel("e-Samsat/UMKM (juta Rp)")
a.set_title("Aktivitas ekonomi vs adopsi pembayaran digital")
plt.tight_layout(); plt.savefig(f"{BASE}/assets/g4_pdrb_vs_adopsi.png", dpi=150); plt.show()
print("korelasi PDRB-adopsi:", round(df_final["pdrb_kapita_2024"].corr(df_final["esamsat_per_umkm"]), 2))
print(df_final.nlargest(2, "esamsat_per_umkm")["nama_kabupaten_kota"].tolist())

# G5 multivariat: matriks korelasi antar indikator
corr = df_final[INDIKATOR].corr()
fig, a = plt.subplots(figsize=(6.5, 5.5))
im = a.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
a.set_xticks(range(len(INDIKATOR)), INDIKATOR, rotation=45, ha="right")
a.set_yticks(range(len(INDIKATOR)), INDIKATOR)
for i in range(len(INDIKATOR)):
    for j in range(len(INDIKATOR)):
        a.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(im, label="korelasi"); a.set_title("Matriks korelasi antar indikator")
plt.tight_layout(); plt.savefig(f"{BASE}/assets/g5_korelasi.png", dpi=150); plt.show()

## Bagian 4 — Rancangan Tabel Bobot Indeks Prioritas (Berbasis ICP Klay)

Bobot mengacu pada Brief ICP Klay (kriteria wilayah terukur + panduan skor: minimal 4 komponen, total 100%, tiap bobot dijustifikasi terhadap ICP):

| Komponen | Indikator Publik | Arah | Bobot | Justifikasi Bisnis (kaitan dengan ICP Klay) |
|---|---|---|---|---|
| Ukuran Pasar | Total UMKM 2023 per kab/kota | + | 30% | Volume leads terbesar; pipeline SDR bergantung pada jumlah calon klien, wilayah kecil tidak mampu menopang target kontak |
| Kepadatan Sektor Target | Share UMKM ritel + F&B + jasa (binaan 2023) | + | 25% | Klay hanya cocok untuk usaha transaksi harian rutin (ritel, F&B, jasa); share tinggi membuat targeting lebih presisi |
| Kesiapan Digital | % penduduk 5+ pengguna internet 2023 | + | 20% | Klay berbasis cloud dengan dashboard analitik; tanpa kesiapan digital produk tidak dapat dimanfaatkan optimal |
| Adopsi Pembayaran Digital | e-Samsat 2024 per UMKM (proksi) | + | 15% | Fitur inti Klay mencakup pembayaran digital (QRIS); dipakai sebagai proksi perilaku non-tunai karena data transaksi QRIS UMKM tidak tersedia secara publik |
| Aktivitas Ekonomi | PDRB per kapita 2024 (ADHB) | + | 10% | Daya beli menunjang kemampuan berlangganan software, tetapi bukan prasyarat utama dibanding empat komponen lain |
| **Total** | | | **100%** | |

Catatan: indeks skor akhir, pengelompokan K-Means, uji sensitivitas bobot, dan Top 5 segmen dikerjakan pada tahap Final Project sesuai batasan scope Milestone 1.

In [ ]:
# BAGIAN 4: VERIFIKASI TABEL BOBOT
BOBOT = {"total_umkm_2023": 30, "kepadatan_icp": 25, "internet_pct_2023": 20,
         "esamsat_per_umkm": 15, "pdrb_kapita_2024": 10}
assert sum(BOBOT.values()) == 100, "total bobot harus 100%"
assert set(BOBOT) == set(INDIKATOR), "komponen bobot harus sama dengan indikator ternormalisasi"
print("Tabel bobot valid: 5 komponen, total", sum(BOBOT.values()), "%")
for k, v in BOBOT.items():
    print(f"  {k:22s} {v}%")